# 前処理強化の比較（train_baseline から複製）

- **ベース**: train_baseline と同じ（category のみ）。
- **強化(TE)**: 8 列を Target Encoding → 「Fresh 率」に近い 1 変数に。（単純な軸で精度が落ちることがある）
- **強化(頻度)**: 8 列を **頻度エンコード**（target は使わない）→ 「出現回数」に変換。別の軸で効かせる。
- **比較**: ベース vs TE vs 頻度 の CV AUC を表示。

In [1]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

from preprocess import load_train_test
from feature_engineering import create_features

print("Setup complete.")

Setup complete.


In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

seed_everything(42)

In [3]:
train, test = load_train_test()
print(f"Train: {len(train):,}, Test: {len(test):,}")

Train: 653,507, Test: 40,716


In [4]:
train = create_features(train)
test = create_features(test)

for col in ["rotten_tomatoes_link", "critic_name", "movie_title", "movie_info", "directors", "authors", "actors", "production_company"]:
    if col in train.columns and col in test.columns:
        train[col] = train[col].fillna("missing").astype(str)
        test[col] = test[col].fillna("missing").astype(str)
for col in ["runtime"]:
    if col in train.columns and col in test.columns and train[col].isna().any():
        med = train[col].median()
        train[col] = train[col].fillna(med)
        test[col] = test[col].fillna(med)

print("特徴量作成完了.")

特徴量作成完了.


In [5]:
FEATURES = [
    "rotten_tomatoes_link", "critic_name", "top_critic", "publisher_name",
    "movie_title", "movie_info", "content_rating", "directors", "authors", "actors",
    "runtime", "production_company",
    "review_year", "review_month", "review_dayofweek",
    "release_year", "release_month", "release_dayofweek", "movie_age_days",
    "genre_Drama", "genre_Comedy", "genre_Action", "genre_Mystery",
    "genre_Fantasy", "genre_Romance", "genre_Horror", "genre_Documentary",
]

# 前処理が粗い 8 列 → TE または 頻度エンコードで置き換え
TE_COLS = ["rotten_tomatoes_link", "critic_name", "movie_title", "movie_info", "directors", "authors", "actors", "production_company"]

FEATURES_BASELINE = FEATURES
FEATURES_IMPROVED = [c + "_te" if c in TE_COLS else c for c in FEATURES]
FEATURES_FREQ = [c + "_freq" if c in TE_COLS else c for c in FEATURES]
print(f"ベース: {len(FEATURES_BASELINE)}列 / TE: {len(FEATURES_IMPROVED)}列 / 頻度: {len(FEATURES_FREQ)}列")

ベース: 27列 / 強化: 27列（TE 置き換え 8列）


## Target Encoding（スムージングあり）

学習データのみでカテゴリ別の目的変数平均を計算し、スムージングで過学習を抑える。CV では各 fold の **学習部分のみ** で fit して val に適用（リーク防止）。

In [6]:
def target_encode_smooth(tr_series, tr_target, te_series, smoothing=10):
    """tr でカテゴリ別平均を計算し、スムージングした値を tr / te に返す。"""
    g = pd.DataFrame({"x": tr_series.astype(str), "y": tr_target}).groupby("x")["y"]
    cnt = g.count()
    mean = g.mean()
    global_mean = tr_target.mean()
    def enc(s):
        s = str(s) if pd.notna(s) else "__nan__"
        if s not in mean.index:
            return global_mean
        c, m = cnt[s], mean[s]
        return (c * m + smoothing * global_mean) / (c + smoothing)
    tr_enc = tr_series.astype(str).map(lambda s: enc(s))
    te_enc = te_series.astype(str).map(lambda s: enc(s))
    return tr_enc.values, te_enc.values

In [7]:
VAL_YEARS = [2013, 2014, 2015, 2016]
time_splits = []
for vy in VAL_YEARS:
    tr_idx = np.where(train["review_year"] < vy)[0]
    val_idx = np.where(train["review_year"] == vy)[0]
    if len(val_idx) > 0:
        time_splits.append((tr_idx, val_idx))
print(f"時系列CV: {len(time_splits)} folds (val years = {VAL_YEARS})")
y = train["target"].values

時系列CV: 4 folds (val years = [2013, 2014, 2015, 2016])


In [8]:
lgb_params = {"objective": "binary", "metric": "auc", "boosting_type": "gbdt",
              "n_estimators": 100, "learning_rate": 0.1, "num_leaves": 31,
              "random_state": 42, "verbosity": -1}

### ① ベース（現状の前処理のみ）で CV

In [9]:
X_base = train[FEATURES_BASELINE].copy()
X_test_base = test[FEATURES_BASELINE].copy()
for c in FEATURES_BASELINE:
    if X_base[c].dtype.name == "object" or str(X_base[c].dtype) == "category":
        X_base[c] = X_base[c].astype("category")
        X_test_base[c] = X_test_base[c].astype("category")

fold_scores_base = []
for fold, (tr_idx, val_idx) in enumerate(time_splits):
    X_tr = X_base.iloc[tr_idx]
    X_val = X_base.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(20, verbose=False)])
    pred = model.predict_proba(X_val)[:, 1]
    fold_scores_base.append(roc_auc_score(y_val, pred))

cv_auc_baseline = np.mean(fold_scores_base)
print(f"ベース（現状前処理のみ） CV AUC (fold mean): {cv_auc_baseline:.4f} ± {np.std(fold_scores_base):.4f}")

ベース（現状前処理のみ） CV AUC (fold mean): 0.7369 ± 0.0044


### ② 強化（TE あり）で CV（fold 内で tr のみ TE fit）

In [10]:
TE_SMOOTHING = 10
fold_scores_imp = []
for fold, (tr_idx, val_idx) in enumerate(time_splits):
    X_tr_imp = train.iloc[tr_idx][FEATURES_BASELINE].copy()
    X_val_imp = train.iloc[val_idx][FEATURES_BASELINE].copy()
    y_tr, y_val = y[tr_idx], y[val_idx]
    for col in TE_COLS:
        tr_enc, val_enc = target_encode_smooth(
            train.iloc[tr_idx][col], y_tr, train.iloc[val_idx][col], smoothing=TE_SMOOTHING
        )
        X_tr_imp[col + "_te"] = tr_enc
        X_val_imp[col + "_te"] = val_enc
    X_tr_f = X_tr_imp[FEATURES_IMPROVED]
    X_val_f = X_val_imp[FEATURES_IMPROVED]
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_tr_f, y_tr, eval_set=[(X_val_f, y_val)], callbacks=[lgb.early_stopping(20, verbose=False)])
    pred = model.predict_proba(X_val_f)[:, 1]
    fold_scores_imp.append(roc_auc_score(y_val, pred))
    print(f"Fold{fold+1}: AUC={fold_scores_imp[-1]:.4f}")

cv_auc_improved = np.mean(fold_scores_imp)
print(f"\n強化（TE あり） CV AUC (fold mean): {cv_auc_improved:.4f} ± {np.std(fold_scores_imp):.4f}")

Fold1: AUC=0.7025
Fold2: AUC=0.7066
Fold3: AUC=0.7010
Fold4: AUC=0.7017

強化（TE あり） CV AUC (fold mean): 0.7030 ± 0.0022


### ③ 強化（頻度エンコード）で CV

**target は使わない**。カテゴリを「学習データ内の出現回数」に変換。例: 映画リンクが 100 回 → 100。val/test で未出現は 0。

In [ ]:
fold_scores_freq = []
for fold, (tr_idx, val_idx) in enumerate(time_splits):
    X_tr_freq = train.iloc[tr_idx][FEATURES_BASELINE].copy()
    X_val_freq = train.iloc[val_idx][FEATURES_BASELINE].copy()
    for col in TE_COLS:
        cnt = train.iloc[tr_idx][col].astype(str).value_counts()
        X_tr_freq[col + "_freq"] = train.iloc[tr_idx][col].astype(str).map(cnt).fillna(0).astype(int)
        X_val_freq[col + "_freq"] = train.iloc[val_idx][col].astype(str).map(cnt).fillna(0).astype(int)
    X_tr_f = X_tr_freq[FEATURES_FREQ]
    X_val_f = X_val_freq[FEATURES_FREQ]
    y_tr, y_val = y[tr_idx], y[val_idx]
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_tr_f, y_tr, eval_set=[(X_val_f, y_val)], callbacks=[lgb.early_stopping(20, verbose=False)])
    pred = model.predict_proba(X_val_f)[:, 1]
    fold_scores_freq.append(roc_auc_score(y_val, pred))
    print(f"Fold{fold+1}: AUC={fold_scores_freq[-1]:.4f}")

cv_auc_freq = np.mean(fold_scores_freq)
print(f"\n強化（頻度エンコード） CV AUC (fold mean): {cv_auc_freq:.4f} ± {np.std(fold_scores_freq):.4f}")

In [11]:
results = pd.DataFrame([
    {"設定": "ベース（現状前処理のみ）", "CV_AUC": round(cv_auc_baseline, 4)},
    {"設定": "強化（TE あり）", "CV_AUC": round(cv_auc_improved, 4)},
    {"設定": "強化（頻度エンコード）", "CV_AUC": round(cv_auc_freq, 4)},
])
results["差分(ベース比)"] = results["CV_AUC"] - cv_auc_baseline
results.loc[0, "差分(ベース比)"] = np.nan
display(results)
print(f"\nベース vs 頻度: {cv_auc_freq - cv_auc_baseline:+.4f}")

,設定,CV_AUC,差分
0,ベース（現状前処理のみ）,0.7369,NaN
1,強化（TE あり）,0.7030,-0.0339



前処理強化による CV AUC の変化: -0.0339


### 提出用（頻度エンコード版）

全 train で出現回数を計算し、頻度特徴で学習 → 予測。ベースより頻度が良ければこちらで提出。

In [ ]:
train_freq = train[FEATURES_BASELINE].copy()
test_freq = test[FEATURES_BASELINE].copy()
for col in TE_COLS:
    cnt = train[col].astype(str).value_counts()
    train_freq[col + "_freq"] = train[col].astype(str).map(cnt).fillna(0).astype(int)
    test_freq[col + "_freq"] = test[col].astype(str).map(cnt).fillna(0).astype(int)

X_freq = train_freq[FEATURES_FREQ]
X_test_freq = test_freq[FEATURES_FREQ]
model_full = lgb.LGBMClassifier(**lgb_params)
model_full.fit(X_freq, y)
final_pred = model_full.predict_proba(X_test_freq)[:, 1]

submission = pd.DataFrame({"ID": test["ID"], "target": final_pred})
submission.to_csv("submission_preprocess_freq.csv", index=False)
print(f"Saved submission_preprocess_freq.csv (rows: {len(submission):,}) [頻度エンコード版]")

importance_df = pd.DataFrame({"feature": FEATURES_FREQ, "importance": model_full.feature_importances_}).sort_values("importance", ascending=False)
display(importance_df)